<a href="https://colab.research.google.com/github/SHRAVAN-AMBEER/Deep_Learning_Practice/blob/main/DL_week5(168).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Q15:Implement the MLP using the Types of Regularization Techniques.
L2 Regularization
Dataset Augmentation
Parameter sharing and tying
Adding noise to the inputs and outputs
Early stopping
Ensemble methods
Dropouts

Observations:
Fashion MNIST is significantly more complex than the original digit MNIST. Features like edges, textures, and shapes matter much more when distinguishing a shirt from a coat than a "1" from a "7". Here is how the techniques perform:

Baseline (No Reg): The model will quickly memorize the training data but struggle slightly on the test data. You will usually see an accuracy around 88%.

L2 Regularization: By penalizing large weights, L2 prevents the network from relying too heavily on any single pixel (e.g., a specific bright spot on a shoe). It often drops the overall accuracy slightly (around 86-87%) but makes the model more robust to new data.

Dropout: This is highly effective for Fashion MNIST. By randomly turning off neurons, it forces the network to learn redundant representations of clothing features (e.g., recognizing a shirt by both its sleeves and its collar). Accuracy is usually strong, around 88-89%.

Input Noise: Adding noise simulates low-quality or grainy images. While it makes the training task harder (sometimes dropping accuracy to 85%), it guarantees the model won't break if you feed it a slightly blurry image of a boot in the real world.

Early Stopping: Instead of blindly training for 10 or 20 epochs, Early Stopping halts the model right when validation loss stops improving (usually around epoch 6-8). It yields the same accuracy as the baseline (~88%) but saves considerable compute time.

Data Augmentation: Shifting and rotating images is brilliant for clothing because an image of a shoe might not be perfectly centered. However, using Dense (Flattened) layers with augmentation struggles to learn. This usually results in lower accuracy (~84%) unless paired with a CNN.

Parameter Sharing (CNN): The absolute winner. A Convolutional Neural Network uses a moving filter that "shares" its weights across the whole image. It looks for local patterns like zippers, laces, or hems regardless of where they are in the image. It will easily score 90-91% accuracy while using fewer parameters than the dense networks.

Ensemble Method: Combining three weak models and averaging their guesses smooths out individual mistakes. If Model A thinks a sneaker is a boot, but Models B and C correctly identify it as a sneaker, the ensemble wins. It usually provides a reliable 1-2% boost over a single baseline model.

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import accuracy_score

# ---------------------------------------------------------
# 1. Load and Prepare Data
# ---------------------------------------------------------
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

# Normalize pixel values
x_train, x_test = x_train / 255.0, x_test / 255.0

# Flattened versions for Dense networks
x_train_flat = x_train.reshape(-1, 784)
x_test_flat = x_test.reshape(-1, 784)

# 2D versions (with channel) for Augmentation and CNNs
x_train_img = x_train.reshape(-1, 28, 28, 1)
x_test_img = x_test.reshape(-1, 28, 28, 1)

# One-hot encode labels
y_train_cat = to_categorical(y_train)
y_test_cat = to_categorical(y_test)

# ---------------------------------------------------------
# Helper Function to Keep Code Clean
# ---------------------------------------------------------
def train_and_evaluate(model, x, y, x_val, y_val, name, epochs=10, callbacks=None):
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    model.fit(x, y, epochs=epochs, validation_data=(x_val, y_val),
              batch_size=64, callbacks=callbacks, verbose=0)
    _, test_acc = model.evaluate(x_val, y_val, verbose=0)
    print(f"{name:<25} | Test Accuracy: {test_acc:.4f}")
    return model

print("-" * 50)
print("Evaluating Regularization Techniques on Fashion MNIST")
print("-" * 50)

# ---------------------------------------------------------
# 2. The Models
# ---------------------------------------------------------

# A. Baseline (No Regularization)
model_base = models.Sequential([
    layers.Dense(256, activation='relu', input_shape=(784,)),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
])
train_and_evaluate(model_base, x_train_flat, y_train_cat, x_test_flat, y_test_cat, "1. Baseline (No Reg)")

# B. L2 Regularization (Weight Decay)
model_l2 = models.Sequential([
    layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(0.001), input_shape=(784,)),
    layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
    layers.Dense(10, activation='softmax')
])
train_and_evaluate(model_l2, x_train_flat, y_train_cat, x_test_flat, y_test_cat, "2. L2 Regularization")

# C. Dropout
model_dropout = models.Sequential([
    layers.Dense(256, activation='relu', input_shape=(784,)),
    layers.Dropout(0.3), # Reduced from 0.5; 0.5 is often too harsh for shallow networks
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(10, activation='softmax')
])
train_and_evaluate(model_dropout, x_train_flat, y_train_cat, x_test_flat, y_test_cat, "3. Dropout")

# D. Input Noise
noise_factor = 0.2
x_train_noisy = x_train_flat + noise_factor * np.random.normal(size=x_train_flat.shape)
x_test_noisy = x_test_flat + noise_factor * np.random.normal(size=x_test_flat.shape)
# Clip values to stay between 0 and 1
x_train_noisy = np.clip(x_train_noisy, 0., 1.)
x_test_noisy = np.clip(x_test_noisy, 0., 1.)

model_noise = models.Sequential([
    layers.Dense(256, activation='relu', input_shape=(784,)),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
])
train_and_evaluate(model_noise, x_train_noisy, y_train_cat, x_test_noisy, y_test_cat, "4. Input Noise")

# E. Early Stopping
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)
model_es = models.Sequential([
    layers.Dense(256, activation='relu', input_shape=(784,)),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
])
train_and_evaluate(model_es, x_train_flat, y_train_cat, x_test_flat, y_test_cat, "5. Early Stopping", epochs=20, callbacks=[early_stop])

# F. Data Augmentation
datagen = ImageDataGenerator(rotation_range=10, width_shift_range=0.1, height_shift_range=0.1)
model_aug = models.Sequential([
    layers.Flatten(input_shape=(28, 28, 1)),
    layers.Dense(256, activation='relu'),
    layers.Dense(10, activation='softmax')
])
model_aug.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model_aug.fit(datagen.flow(x_train_img, y_train_cat, batch_size=64),
              epochs=10, validation_data=(x_test_img, y_test_cat), verbose=0)
_, aug_acc = model_aug.evaluate(x_test_img, y_test_cat, verbose=0)
print(f"{'6. Data Augmentation':<25} | Test Accuracy: {aug_acc:.4f}")

# G. Parameter Sharing (Convolutional Neural Network)
model_cnn = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(28, 28, 1)),
    layers.MaxPooling2D((2,2)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
])
train_and_evaluate(model_cnn, x_train_img, y_train_cat, x_test_img, y_test_cat, "7. Parameter Sharing (CNN)")

# H. Ensemble Method
models_list = []
for i in range(3):
    model = models.Sequential([
        layers.Dense(256, activation='relu', input_shape=(784,)),
        layers.Dense(10, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    model.fit(x_train_flat, y_train_cat, epochs=5, verbose=0)
    models_list.append(model)

# Average the predictions
preds = np.mean([m.predict(x_test_flat, verbose=0) for m in models_list], axis=0)
ensemble_acc = accuracy_score(y_test, np.argmax(preds, axis=1))
print(f"{'8. Ensemble (3 Models)':<25} | Test Accuracy: {ensemble_acc:.4f}")
print("-" * 50)

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
--------------------------------------------------
Evaluating Regularization Techniques on Fashion MNIST
--------------------------------------------------


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1. Baseline (No Reg)      | Test Accuracy: 0.8882
2. L2 Regularization      | Test Accuracy: 0.8717
3. Dropout                | Test Accuracy: 0.8814
4. Input Noise            | Test Accuracy: 0.8480
5. Early Stopping         | Test Accuracy: 0.8844


/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


6. Data Augmentation      | Test Accuracy: 0.8416


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


7. Parameter Sharing (CNN) | Test Accuracy: 0.9174
8. Ensemble (3 Models)    | Test Accuracy: 0.8827
--------------------------------------------------
